In [ ]:
# imports
import tensorflow as tf
from tensorflow import keras as k
from tensorflow.keras import layers as l
from tensorflow.keras import mixed_precision

import numpy as np
import pandas as pd
import os, shutil

from sklearn.metrics import classification_report as cr
from sklearn.metrics import confusion_matrix as cm

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# config
seed=42
batch=64
img_size=(96,96)
AUTOTUNE=tf.data.AUTOTUNE

tf.random.set_seed(42)
np.random.seed(42)

mixed_precision.set_global_policy('mixed_float16')

In [ ]:
# data
!rm -rf tiny-imagenet-200
!wget -q http://cs231n.stanford.edu/tiny-imagenet-200.zip
!unzip -oq tiny-imagenet-200.zip

# validation organization
val_dir="/content/tiny-imagenet-200/val"
img_dir=os.path.join(val_dir,"images")
anno_file=os.path.join(val_dir,"val_annotations.txt")

if os.path.exists(img_dir):

    with open(anno_file) as f:

        for line in f:

            img,cls=line.strip().split('\t')[:2]

            cls_path=os.path.join(val_dir,cls)
            os.makedirs(cls_path,exist_ok=True)

            src=os.path.join(img_dir,img)
            dst=os.path.join(cls_path,img)

            if os.path.exists(src):
                shutil.move(src,dst)

    shutil.rmtree(img_dir)

    print("validation organized")

else:
    print("it was already organized")

validation organized


In [ ]:
# datasets
train_ds=tf.keras.preprocessing.image_dataset_from_directory(
    "/content/tiny-imagenet-200/train",
    image_size=img_size,
    batch_size=batch,
    shuffle=True,
    seed=seed
)

val_ds=tf.keras.preprocessing.image_dataset_from_directory(
    "/content/tiny-imagenet-200/val",
    image_size=img_size,
    batch_size=batch,
    shuffle=False
)

Found 100000 files belonging to 200 classes.
Found 10000 files belonging to 200 classes.


In [ ]:
# performance
train_ds=(
    train_ds
    .shuffle(1000)
    .prefetch(AUTOTUNE)
)

val_ds=(
    val_ds
    .prefetch(AUTOTUNE)
)

In [ ]:
# callbacks
cb=[
    k.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True
    ),

    k.callbacks.ReduceLROnPlateau(
        patience=2,
        factor=0.3,
        min_lr=1e-7
    ),

    k.callbacks.ModelCheckpoint(
        "best_model.keras",
        save_best_only=True,
        monitor="val_accuracy"
    )
]

In [ ]:
# mobilenetv2
base_mob=k.applications.MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(96,96,3)
)

base_mob.trainable=False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# augmentation
aug_mob=k.Sequential([
    l.RandomFlip('horizontal'),
    l.RandomRotation(0.1),
    l.RandomZoom(0.1),
    l.RandomContrast(0.1),
])

# model
model_mob=k.Sequential([
    l.Input(shape=(96,96,3)),
    aug_mob,
    l.Lambda(k.applications.mobilenet_v2.preprocess_input),
    base_mob,

    l.GlobalAveragePooling2D(),
    l.Dense(256,activation='relu'),
    l.BatchNormalization(),
    l.Dropout(0.4),
    l.Dense(200,activation='softmax',dtype='float32')
])

model_mob.compile(
    optimizer=k.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# head training
hist_mob_head=model_mob.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    verbose=2
)

Epoch 1/10
1563/1563 - 128s - 82ms/step - accuracy: 0.3868 - loss: 2.7471 - val_accuracy: 0.6106 - val_loss: 1.6061
Epoch 2/10
1563/1563 - 75s - 48ms/step - accuracy: 0.4766 - loss: 2.2129 - val_accuracy: 0.6218 - val_loss: 1.5387
Epoch 3/10
1563/1563 - 66s - 42ms/step - accuracy: 0.4928 - loss: 2.1213 - val_accuracy: 0.6285 - val_loss: 1.4904
Epoch 4/10
1563/1563 - 61s - 39ms/step - accuracy: 0.5022 - loss: 2.0775 - val_accuracy: 0.6323 - val_loss: 1.4833
Epoch 5/10
1563/1563 - 61s - 39ms/step - accuracy: 0.5076 - loss: 2.0514 - val_accuracy: 0.6373 - val_loss: 1.4730
Epoch 6/10
1563/1563 - 61s - 39ms/step - accuracy: 0.5150 - loss: 2.0125 - val_accuracy: 0.6360 - val_loss: 1.4611
Epoch 7/10
1563/1563 - 60s - 38ms/step - accuracy: 0.5199 - loss: 1.9847 - val_accuracy: 0.6384 - val_loss: 1.4413
Epoch 8/10
1563/1563 - 60s - 38ms/step - accuracy: 0.5224 - loss: 1.9737 - val_accuracy: 0.6394 - val_loss: 1.4425
Epoch 9/10
1563/1563 - 60s - 38ms/step - accuracy: 0.5278 - loss: 1.9483 - val_

In [ ]:
# unfreeze
base_mob.trainable=True

for layer in base_mob.layers[:-40]:
    layer.trainable=False

# fine-tuning
model_mob.compile(
    optimizer=k.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# fit
hist_mob_ft=model_mob.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    verbose=2,
    callbacks=cb
)

# evaluate
mob_loss,mob_acc=model_mob.evaluate(val_ds)

# save
model_mob.save('mobilenetv2_tinyimagenet.keras')

Epoch 1/10
1563/1563 - 113s - 72ms/step - accuracy: 0.4981 - loss: 2.1203 - val_accuracy: 0.6530 - val_loss: 1.4045 - learning_rate: 1.0000e-05
Epoch 2/10
1563/1563 - 82s - 53ms/step - accuracy: 0.5313 - loss: 1.9386 - val_accuracy: 0.6580 - val_loss: 1.3698 - learning_rate: 1.0000e-05
Epoch 3/10
1563/1563 - 84s - 54ms/step - accuracy: 0.5448 - loss: 1.8594 - val_accuracy: 0.6581 - val_loss: 1.3615 - learning_rate: 1.0000e-05
Epoch 4/10
1563/1563 - 84s - 53ms/step - accuracy: 0.5562 - loss: 1.8059 - val_accuracy: 0.6630 - val_loss: 1.3425 - learning_rate: 1.0000e-05
Epoch 5/10
1563/1563 - 82s - 53ms/step - accuracy: 0.5670 - loss: 1.7546 - val_accuracy: 0.6657 - val_loss: 1.3340 - learning_rate: 1.0000e-05
Epoch 6/10
1563/1563 - 83s - 53ms/step - accuracy: 0.5757 - loss: 1.7160 - val_accuracy: 0.6686 - val_loss: 1.3159 - learning_rate: 1.0000e-05
Epoch 7/10
1563/1563 - 81s - 52ms/step - accuracy: 0.5806 - loss: 1.6897 - val_accuracy: 0.6715 - val_loss: 1.3049 - learning_rate: 1.0000e-0

In [ ]:
# resnet50
base_res=k.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(96,96,3)
)

base_res.trainable=False

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
aug_res=k.Sequential([
    l.RandomFlip('horizontal'),
    l.RandomRotation(0.1),
    l.RandomZoom(0.1),
    l.RandomContrast(0.1)
])

model_res=k.Sequential([
    l.Input(shape=(96,96,3)),
    aug_res,
    l.Lambda(k.applications.resnet50.preprocess_input),
    base_res,

    l.GlobalAveragePooling2D(),
    l.Dense(256,activation='relu'),
    l.BatchNormalization(),
    l.Dropout(0.4),
    l.Dense(200,activation='softmax',dtype='float32')
])

model_res.compile(
    optimizer=k.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# head training
hist_res_head=model_res.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    verbose=2
)

Epoch 1/10
1563/1563 - 95s - 61ms/step - accuracy: 0.3876 - loss: 2.7029 - val_accuracy: 0.5503 - val_loss: 1.8257
Epoch 2/10
1563/1563 - 83s - 53ms/step - accuracy: 0.4693 - loss: 2.2175 - val_accuracy: 0.5683 - val_loss: 1.7392
Epoch 3/10
1563/1563 - 82s - 52ms/step - accuracy: 0.4894 - loss: 2.1098 - val_accuracy: 0.5833 - val_loss: 1.6865
Epoch 4/10
1563/1563 - 81s - 52ms/step - accuracy: 0.5000 - loss: 2.0528 - val_accuracy: 0.5826 - val_loss: 1.6691
Epoch 5/10
1563/1563 - 82s - 53ms/step - accuracy: 0.5098 - loss: 2.0023 - val_accuracy: 0.5873 - val_loss: 1.6524
Epoch 6/10
1563/1563 - 82s - 52ms/step - accuracy: 0.5124 - loss: 1.9734 - val_accuracy: 0.5905 - val_loss: 1.6521
Epoch 7/10
1563/1563 - 82s - 52ms/step - accuracy: 0.5212 - loss: 1.9410 - val_accuracy: 0.5961 - val_loss: 1.6270
Epoch 8/10
1563/1563 - 82s - 53ms/step - accuracy: 0.5281 - loss: 1.9103 - val_accuracy: 0.5993 - val_loss: 1.6121
Epoch 9/10
1563/1563 - 81s - 52ms/step - accuracy: 0.5303 - loss: 1.8900 - val_a

In [ ]:
# unfreeze
base_res.trainable=True

for layer in base_res.layers[:-40]:
    layer.trainable=False

# fine-tuning
model_res.compile(
    optimizer=k.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# fit
hist_res_ft=model_res.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    verbose=2,
    callbacks=cb
)

# evaluate
res_loss,res_acc=model_res.evaluate(val_ds)

# save
model_res.save('resnet50_tinyimagenet.keras')

Epoch 1/10
1563/1563 - 138s - 88ms/step - accuracy: 0.5360 - loss: 1.8777 - val_accuracy: 0.6159 - val_loss: 1.5682 - learning_rate: 1.0000e-05
Epoch 2/10
1563/1563 - 121s - 77ms/step - accuracy: 0.5734 - loss: 1.6945 - val_accuracy: 0.6267 - val_loss: 1.5085 - learning_rate: 1.0000e-05
Epoch 3/10
1563/1563 - 116s - 74ms/step - accuracy: 0.5922 - loss: 1.6087 - val_accuracy: 0.6319 - val_loss: 1.4907 - learning_rate: 3.0000e-06
Epoch 4/10
1563/1563 - 116s - 74ms/step - accuracy: 0.6004 - loss: 1.5712 - val_accuracy: 0.6341 - val_loss: 1.4773 - learning_rate: 3.0000e-06
Epoch 5/10
1563/1563 - 116s - 74ms/step - accuracy: 0.6041 - loss: 1.5512 - val_accuracy: 0.6339 - val_loss: 1.4735 - learning_rate: 9.0000e-07
157/157 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - accuracy: 0.6159 - loss: 1.5682


In [ ]:
# efficientnetb0
base_efc=k.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(96,96,3)
)

base_efc.trainable=False

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
aug_efc=k.Sequential([
    l.RandomFlip('horizontal'),
    l.RandomRotation(0.1),
    l.RandomZoom(0.1),
    l.RandomContrast(0.1)
])

model_efc=k.Sequential([
    l.Input(shape=(96,96,3)),
    aug_efc,
    l.Lambda(k.applications.efficientnet.preprocess_input),
    base_efc,

    l.GlobalAveragePooling2D(),
    l.Dense(256,activation='relu'),
    l.BatchNormalization(),
    l.Dropout(0.4),
    l.Dense(200,activation='softmax',dtype='float32')
])

model_efc.compile(
    optimizer=k.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# head training
hist_efc_head=model_efc.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    verbose=2
)

Epoch 1/10
1563/1563 - 148s - 94ms/step - accuracy: 0.3833 - loss: 2.8017 - val_accuracy: 0.5723 - val_loss: 1.7516
Epoch 2/10
1563/1563 - 87s - 55ms/step - accuracy: 0.4682 - loss: 2.2657 - val_accuracy: 0.5908 - val_loss: 1.6659
Epoch 3/10
1563/1563 - 83s - 53ms/step - accuracy: 0.4877 - loss: 2.1669 - val_accuracy: 0.5978 - val_loss: 1.6383
Epoch 4/10
1563/1563 - 82s - 52ms/step - accuracy: 0.4970 - loss: 2.1022 - val_accuracy: 0.6059 - val_loss: 1.6167
Epoch 5/10
1563/1563 - 79s - 51ms/step - accuracy: 0.5078 - loss: 2.0594 - val_accuracy: 0.6074 - val_loss: 1.6040
Epoch 6/10
1563/1563 - 79s - 50ms/step - accuracy: 0.5100 - loss: 2.0332 - val_accuracy: 0.6098 - val_loss: 1.6073
Epoch 7/10
1563/1563 - 78s - 50ms/step - accuracy: 0.5156 - loss: 2.0077 - val_accuracy: 0.6088 - val_loss: 1.5908
Epoch 8/10
1563/1563 - 84s - 54ms/step - accuracy: 0.5225 - loss: 1.9843 - val_accuracy: 0.6102 - val_loss: 1.5983
Epoch 9/10
1563/1563 - 85s - 55ms/step - accuracy: 0.5237 - loss: 1.9664 - val_

In [ ]:
# unfreeze
base_efc.trainable=True

for layer in base_efc.layers[:-40]:
    layer.trainable=False

# fine-tuning
model_efc.compile(
    optimizer=k.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# fit
hist_efc_ft=model_efc.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    verbose=2,
    callbacks=cb
)

# evaluate
efc_loss,efc_acc=model_efc.evaluate(val_ds)

# save
model_efc.save('efficientnetb0_tinyimagenet.keras')

Epoch 1/10
1563/1563 - 145s - 93ms/step - accuracy: 0.4694 - loss: 2.2989 - val_accuracy: 0.5991 - val_loss: 1.7078 - learning_rate: 1.0000e-05
Epoch 2/10
1563/1563 - 102s - 65ms/step - accuracy: 0.5084 - loss: 2.0796 - val_accuracy: 0.6140 - val_loss: 1.6149 - learning_rate: 1.0000e-05
Epoch 3/10
1563/1563 - 96s - 61ms/step - accuracy: 0.5216 - loss: 2.0033 - val_accuracy: 0.6157 - val_loss: 1.5997 - learning_rate: 3.0000e-06
Epoch 4/10
1563/1563 - 96s - 61ms/step - accuracy: 0.5245 - loss: 1.9831 - val_accuracy: 0.6187 - val_loss: 1.5808 - learning_rate: 3.0000e-06
Epoch 5/10
1563/1563 - 95s - 61ms/step - accuracy: 0.5253 - loss: 1.9648 - val_accuracy: 0.6198 - val_loss: 1.5761 - learning_rate: 9.0000e-07
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - accuracy: 0.5991 - loss: 1.7078


In [ ]:
# histories
pd.DataFrame(hist_mob_ft.history).to_csv(
    'mobilenet_history.csv',
    index=False
)

pd.DataFrame(hist_res_ft.history).to_csv(
    'resnet_history.csv',
    index=False
)

pd.DataFrame(hist_efc_ft.history).to_csv(
    'efficientnet_history.csv',
    index=False
)

# final dataframe
results=[]

for name,acc,loss in [
    ('mobilenetv2',mob_acc,mob_loss),
    ('resnet50',res_acc,res_loss),
    ('efficientnetb0',efc_acc,efc_loss),
]:

    results.append([
        name,
        acc,
        loss
    ])

df=pd.DataFrame(
    results,
    columns=[
        'model',
        'val_accuracy',
        'val_loss'
    ]
)

print(df)

df.to_csv(
    'results.csv',
    index=False
)

            model  val_accuracy  val_loss
0     mobilenetv2        0.6769  1.291077
1        resnet50        0.6159  1.568211
2  efficientnetb0        0.5991  1.707814


In [ ]:
# downloads
from google.colab import files

files.download('mobilenetv2_tinyimagenet.keras')
files.download('resnet50_tinyimagenet.keras')
files.download('efficientnetb0_tinyimagenet.keras')

files.download('mobilenet_history.csv')
files.download('resnet_history.csv')
files.download('efficientnet_history.csv')

files.download('results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>